<a href="https://colab.research.google.com/github/sabdaaf/analisis-sentimen-twitter-svm-dt-indobert/blob/main/sentimen_analisis_x_NB_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ANALISIS SENTIMEN #KABURAJADULU TWITTER/X**

Pipeline: Cleaning -> Labeling (Lexicon-based) -> Feature Extraction
          (TF-IDF & IndoBERT Embedding) -> Training (SVM & Decision Tree)
          -> Evaluasi & Perbandingan

In [3]:
!pip install Sastrawi

#IMPORTS & CONFIG
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix)

# Sastrawi untuk stopword removal & stemming (khusus jalur TF-IDF)
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [4]:
CONFIG = {
    "input_path": "scrapingTwitter.xlsx",
    "text_col": "text",
    "id_col": "id",
    "random_state": 42,
    "test_size": 0.2,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device yang dipakai: {DEVICE}")

Device yang dipakai: cpu


**load data**

In [5]:
#DATA LOADING
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, engine="openpyxl")
    print(f"Data awal: {df.shape[0]} baris, {df.shape[1]} kolom")
    return df

In [6]:
df = load_data(CONFIG["input_path"])

Data awal: 11940 baris, 32 kolom


In [7]:
df.head()

,type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,...,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote
0,tweet,2086493671093141658,https://x.com/KuliHarian82490/status/208649367...,https://twitter.com/KuliHarian82490/status/208...,Kabur aja dulu\n\nKalian tertarik kerja luar n...,NaN,0,0,0,0,...,NaN,{},{'media': [{'display_url': 'pic.x.com/jJGu7AK5...,NaN,NaN,NaN,NaN,False,0,NaN
1,tweet,2086458790652870946,https://x.com/eufrasiarts/status/2086458790652...,https://twitter.com/eufrasiarts/status/2086458...,@changaaa13 kayaknya plan kabur aja dulu kali ...,NaN,0,0,0,0,...,NaN,{},{'user_mentions': [{'id_str': '124079726645089...,NaN,NaN,NaN,NaN,False,0,NaN
2,tweet,2086457445136904207,https://x.com/ivaniskand43902/status/208645744...,https://twitter.com/ivaniskand43902/status/208...,"@azmisabriii yu bang urang kabur aja dulu,,😁",NaN,0,0,0,0,...,NaN,{},{'user_mentions': [{'id_str': '124377590804698...,NaN,NaN,NaN,NaN,False,0,NaN
3,tweet,2086453073380360208,https://x.com/dahayudunia/status/2086453073380...,https://twitter.com/dahayudunia/status/2086453...,"biarin gais lucu-lucu dulu, sebelum.. (kabur) ...",NaN,0,1,3,0,...,"{'binding_values': [{'key': 'thumbnail_image',...",{},{'urls': [{'display_url': 'alterspring.org/@ma...,NaN,NaN,NaN,NaN,False,0,NaN
4,tweet,2086396383263596823,https://x.com/Vijeland/status/2086396383263596823,https://twitter.com/Vijeland/status/2086396383...,"Bener juga @Milkchoco999, buset dah pelayannya...",NaN,0,1,2,0,...,NaN,{},{'user_mentions': [{'id_str': '171520811772481...,NaN,NaN,"{'type': 'tweet', 'id': '2085934447787209115',...",NaN,False,0,1.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11940 entries, 0 to 11939
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   type                      11940 non-null  object 
 1   id                        11940 non-null  int64  
 2   url                       11940 non-null  object 
 3   twitterUrl                11940 non-null  object 
 4   text                      11940 non-null  object 
 5   source                    0 non-null      float64
 6   retweetCount              11940 non-null  int64  
 7   replyCount                11940 non-null  int64  
 8   likeCount                 11940 non-null  int64  
 9   quoteCount                11940 non-null  int64  
 10  viewCount                 11940 non-null  int64  
 11  createdAt                 11940 non-null  object 
 12  lang                      11940 non-null  object 
 13  bookmarkCount             11940 non-null  int64  
 14  isRepl

In [9]:
print(df.isnull().sum())

type                            0
id                              0
url                             0
twitterUrl                      0
text                            0
source                      11940
retweetCount                    0
replyCount                      0
likeCount                       0
quoteCount                      0
viewCount                       0
createdAt                       0
lang                            0
bookmarkCount                   0
isReply                         0
inReplyToId                  6603
conversationId                  0
inReplyToUserId             11940
inReplyToUsername           11940
isPinned                        0
author                          0
extendedEntities                0
card                        11579
place                           0
entities                        0
reply_to_user_results       11940
quoted_tweet_results        11940
quoted_tweet                 9591
retweeted_tweet             11940
isConversation

In [10]:
print(df.duplicated().sum())

0


**proprocessing**

In [11]:
#TEXT CLEANING
def clean_noise(text: str) -> str:
    """Hapus URL, mention, hashtag simbol, newline, whitespace berlebih."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"@\w+", "", text)                    # mention
    text = re.sub(r"#(\w+)", r"\1", text)                # simbol # dibuang, teks disimpan
    text = re.sub(r"[\r\n]+", " ", text)                 # newline
    text = re.sub(r"[^\w\s,.!?]", " ", text)             # emoji & simbol aneh
    text = re.sub(r"\s+", " ", text).strip()
    return text


def case_folding(text: str) -> str:
    return text.lower()

In [12]:
# Kamus slang — silakan perluas dengan kamus alay yang lebih lengkap
# (mis. https://github.com/nasalsabila/kamus-alay) untuk hasil lebih baik
SLANG_DICT = {
    "bgt": "banget", "yg": "yang", "gw": "saya", "gua": "saya",
    "lu": "kamu", "pake": "pakai", "krn": "karena", "gk": "tidak",
    "ga": "tidak", "gak": "tidak", "sampe": "sampai", "tp": "tapi",
    "dr": "dari", "utk": "untuk", "sm": "sama", "jd": "jadi",
    "udah": "sudah", "udh": "sudah", "blm": "belum", "dgn": "dengan",
}

In [13]:

def normalize_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    words = text.split()
    return " ".join(slang_dict.get(w, w) for w in words)


In [14]:
def preprocess_base(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Tahap 2-3: cleaning + case folding + normalisasi slang + drop kosong/duplikat."""
    df = df[[CONFIG["id_col"], text_col]].copy()
    df["text_clean"] = df[text_col].apply(clean_noise)
    df["text_clean"] = df["text_clean"].apply(case_folding)
    df["text_clean"] = df["text_clean"].apply(normalize_slang)

    before = len(df)
    df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)
    df = df.drop_duplicates(subset="text_clean").reset_index(drop=True)
    print(f"Baris dibuang (kosong/duplikat): {before - len(df)}")
    print(f"Ukuran data setelah preprocessing dasar: {df.shape}")
    return df

In [15]:
def load_lexicon(pos_path: str, neg_path: str) -> dict:
    pos_df = pd.read_csv(pos_path, sep="\t")
    neg_df = pd.read_csv(neg_path, sep="\t")
    lexicon = {}
    lexicon.update(dict(zip(pos_df["word"], pos_df["weight"])))
    lexicon.update(dict(zip(neg_df["word"], neg_df["weight"])))
    return lexicon

In [16]:
#[5A] JALUR TF-IDF (untuk SVM & Decision Tree — versi klasik)
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()


def preprocess_for_tfidf(text: str) -> str:
    text = stopword_remover.remove(text)
    text = stemmer.stem(text)
    return text


def build_tfidf_features(df: pd.DataFrame, max_features: int = 5000):
    df["text_tfidf_ready"] = df["text_clean"].apply(preprocess_for_tfidf)
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    X_tfidf = vectorizer.fit_transform(df["text_tfidf_ready"])
    print(f"Dimensi fitur TF-IDF: {X_tfidf.shape}")
    return X_tfidf, vectorizer

In [17]:
# [5B] JALUR INDOBERT EMBEDDING (untuk SVM & Decision Tree — versi kontekstual)
# Catatan: teks TIDAK di-stopword-removal / di-stem di jalur ini,
# karena BERT butuh struktur kalimat utuh untuk konteks yang baik.
def load_indobert(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def get_bert_embeddings(text_list, tokenizer, model, batch_size=16, max_length=128):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = list(text_list[i:i + batch_size])
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=max_length, return_tensors="pt"
            ).to(DEVICE)
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # token [CLS]
            embeddings.append(cls_embeddings.cpu())
            if (i // batch_size) % 20 == 0:
                print(f"  Embedding batch {i // batch_size + 1}/"
                      f"{(len(text_list) - 1) // batch_size + 1}")
    return torch.cat(embeddings, dim=0).numpy()

In [18]:
#TRAIN/TEST SPLIT (dipakai sama untuk kedua jalur fitur agar fair)
def split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

In [19]:
# TRAINING & TUNING
def train_svm(X_train, y_train, cv=5):
    param_grid = {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]}
    grid = GridSearchCV(
        SVC(class_weight="balanced", random_state=CONFIG["random_state"]),
        param_grid, cv=StratifiedKFold(cv), scoring="f1_macro", n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best SVM params: {grid.best_params_}")
    return grid.best_estimator_


def train_decision_tree(X_train, y_train, cv=5):
    param_grid = {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10]}
    grid = GridSearchCV(
        DecisionTreeClassifier(class_weight="balanced", random_state=CONFIG["random_state"]),
        param_grid, cv=StratifiedKFold(cv), scoring="f1_macro", n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best Decision Tree params: {grid.best_params_}")
    return grid.best_estimator_

In [20]:
# EVALUASI
def evaluate_model(model, X_test, y_test, model_name: str):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    print(f"\n=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-macro : {f1_macro:.4f}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Prediksi")
    plt.ylabel("Aktual")
    plt.tight_layout()
    plt.savefig(f"confmat_{model_name.replace(' ', '_')}.png")
    plt.close()

    return {"model": model_name, "accuracy": acc, "f1_macro": f1_macro}

In [21]:
#MAIN PIPELINE
def main():
    # --- Tahap 1-3: Load & preprocessing dasar ---
    df_raw = load_data(CONFIG["input_path"])
    df = preprocess_base(df_raw, CONFIG["text_col"])

    # --- Tahap 4: Labeling sentimen ---
    # Sesuaikan path ke file lexicon InSet hasil unduhan
    lexicon = load_lexicon("lexicon_positive.tsv", "lexicon_negative.tsv")
    df = apply_lexicon_labeling(df, lexicon)

    # >>> Titik berhenti disarankan: cek/koreksi df['label'] secara manual
    # df.to_csv("labeled_data_for_review.csv", index=False)

    y = df["label"]

    # --- Tahap 5A: Fitur TF-IDF ---
    X_tfidf, tfidf_vectorizer = build_tfidf_features(df)

    # --- Tahap 5B: Fitur IndoBERT embedding ---
    tokenizer, bert_model = load_indobert(CONFIG["indobert_model"])
    X_bert = get_bert_embeddings(
        df["text_clean"].tolist(), tokenizer, bert_model,
        batch_size=CONFIG["bert_batch_size"], max_length=CONFIG["bert_max_length"]
    )

    results = []

    # --- Tahap 6-8: Split, training, evaluasi — Jalur TF-IDF ---
    X_train, X_test, y_train, y_test = split_data(
        X_tfidf, y, CONFIG["test_size"], CONFIG["random_state"]
    )
    svm_tfidf = train_svm(X_train, y_train)
    results.append(evaluate_model(svm_tfidf, X_test, y_test, "SVM_TFIDF"))

    dt_tfidf = train_decision_tree(X_train, y_train)
    results.append(evaluate_model(dt_tfidf, X_test, y_test, "DecisionTree_TFIDF"))

    # --- Tahap 6-8: Split, training, evaluasi — Jalur IndoBERT ---
    X_train_b, X_test_b, y_train_b, y_test_b = split_data(
        X_bert, y, CONFIG["test_size"], CONFIG["random_state"]
    )
    svm_bert = train_svm(X_train_b, y_train_b)
    results.append(evaluate_model(svm_bert, X_test_b, y_test_b, "SVM_IndoBERT"))

    dt_bert = train_decision_tree(X_train_b, y_train_b)
    results.append(evaluate_model(dt_bert, X_test_b, y_test_b, "DecisionTree_IndoBERT"))

    # --- Tahap 9: Ringkasan perbandingan ---
    results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
    print("\n=== RINGKASAN PERBANDINGAN MODEL ===")
    print(results_df.to_string(index=False))
    results_df.to_csv("model_comparison_results.csv", index=False)

    return df, results_df


if __name__ == "__main__":
    df_final, results_df = main()


Data awal: 11940 baris, 32 kolom
Baris dibuang (kosong/duplikat): 524
Ukuran data setelah preprocessing dasar: (11416, 3)


FileNotFoundError: [Errno 2] No such file or directory: 'lexicon_positive.tsv'

In [23]:
  # --- Tahap 6-8: Split, training, evaluasi — Jalur TF-IDF ---
    X_train, X_test, y_train, y_test = split_data(
        X_tfidf, y, CONFIG["test_size"], CONFIG["random_state"]
    )
    svm_tfidf = train_svm(X_train, y_train)
    results.append(evaluate_model(svm_tfidf, X_test, y_test, "SVM_TFIDF"))

    dt_tfidf = train_decision_tree(X_train, y_train)
    results.append(evaluate_model(dt_tfidf, X_test, y_test, "DecisionTree_TFIDF"))

    # --- Tahap 6-8: Split, training, evaluasi — Jalur IndoBERT ---
    X_train_b, X_test_b, y_train_b, y_test_b = split_data(
        X_bert, y, CONFIG["test_size"], CONFIG["random_state"]
    )
    svm_bert = train_svm(X_train_b, y_train_b)
    results.append(evaluate_model(svm_bert, X_test_b, y_test_b, "SVM_IndoBERT"))

    dt_bert = train_decision_tree(X_train_b, y_train_b)
    results.append(evaluate_model(dt_bert, X_test_b, y_test_b, "DecisionTree_IndoBERT"))



IndentationError: unexpected indent (2334882581.py, line 2)

In [24]:

    # --- Tahap 9: Ringkasan perbandingan ---
    results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
    print("\n=== RINGKASAN PERBANDINGAN MODEL ===")
    print(results_df.to_string(index=False))
    results_df.to_csv("model_comparison_results.csv", index=False)

    return df, results_df


if __name__ == "__main__":
    df_final, results_df = main()

IndentationError: expected an indented block after 'if' statement on line 10 (3302775712.py, line 11)